# International Football Score Prediction

**Dataset:** `martj42/international_results` · matches from 2018-01-01 onward  
**Demo fixture:** Argentina vs Austria, 2026-06-22 (FIFA World Cup, neutral venue)  
**Stack:** Python 3.12+, Polars, NumPy, SciPy, PyMC, ArviZ, XGBoost, Optuna, Scikit-Learn, Matplotlib, Seaborn

---

## Methodology Overview

The pipeline produces a full **(K+1) × (K+1) scoreline probability matrix** for any fixture by combining three independent estimators of each team's home and away goal rates:

| Stage | Method | Contribution |
|---|---|---|
| A | Dixon-Coles (weighted MAP via SciPy) | Frequentist baseline; source of static team-strength features; only stage with the explicit low-score `tau` correlation correction |
| B | Bayesian hierarchical Poisson (PyMC / NUTS) | Full posterior over attack, defense, home-advantage, and intercept; posterior-predictive scoreline distribution for the target fixture; R-hat / ESS convergence diagnostics mandatory |
| C | XGBoost (two Poisson-objective regressors, Optuna-tuned) | Nonlinear correction layer over Dixon-Coles strength features and rolling-form features |

The three matrices are combined into a single ensemble matrix via **backtest-loss-weighted averaging** (softmax over negative log loss from the chronological holdout).  

The notebook runs in two passes:  
- **Backtest pass** — all three models are fit on `train` (before cutoff), scored on `test` (after cutoff), and used to derive ensemble weights and evaluate per-model metrics.  
- **Production pass** — all three models are refit on the full played history to give the best possible current ratings, then used to predict the headline fixture.

---

## Module Map

```
config            Configuration dataclasses and constants
data_ingestion    Raw CSV fetch, schema validation, caching
team_registry     FIFA-code resolution, core-team selection
data_preparation  Played/upcoming split, date filtering, weighting, chronological split
dixon_coles       Weighted MAP fit (attack / defense / home-advantage / rho)
feature_eng       Leak-free rolling form, static DC features, fixture feature row
bayesian_model    PyMC model build/fit, convergence diagnostics, posterior extraction
xgboost_model     Training, Optuna search, prediction
scoring           Poisson/DC scoreline matrix construction, outcome collapse, top-N
ensemble          Backtest-weighted matrix combination, entropy
evaluation        Backtest metrics, calibration, baseline comparison
visualization     Heatmap, outcome bar chart, top-10 bar chart, diagnostic plots
```

Module dependencies run strictly top-to-bottom in the list above. `config` has no internal dependencies and is imported everywhere.

---

## Section 1 — Configuration

All pipeline behaviour is controlled by a single frozen `PipelineConfig` dataclass composed of twelve nested config objects. No magic numbers appear anywhere else in the notebook; every tunable value is a named field here.

**Design rule:** `PipelineConfig` is instantiated once in subsection 1.14 and passed explicitly to every top-level function. Global mutable state is forbidden.

### 1.1 Imports

All third-party and standard-library imports for the entire notebook are declared here, in a single cell, to make dependency auditing straightforward.

In [ ]:
# ── Standard library ────────────────────────────────────────────────────────
import datetime
import itertools
import pathlib
import uuid
import warnings
from dataclasses import dataclass, field
from typing import Iterator

# ── Numerics / data ──────────────────────────────────────────────────────────
import numpy as np
import polars as pl

# ── Scientific / statistical ─────────────────────────────────────────────────
import scipy.optimize as opt
import scipy.special as special
import scipy.stats as stats

# ── Bayesian ─────────────────────────────────────────────────────────────────
import arviz as az
import pymc as pm

# ── Machine learning ─────────────────────────────────────────────────────────
import optuna
import xgboost as xgb
from sklearn.metrics import log_loss as sk_log_loss, brier_score_loss

# ── Visualisation ────────────────────────────────────────────────────────────
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

matplotlib.use("Agg")          # non-interactive backend; change to "TkAgg" for live display
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)

# ── Suppress noisy but non-critical log output ────────────────────────────────
warnings.filterwarnings("ignore", category=UserWarning, module="pymc")
warnings.filterwarnings("ignore", category=FutureWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Reproducibility guard: verify version strings ─────────────────────────────
_DEPS: dict[str, str] = {
    "numpy": np.__version__,
    "polars": pl.__version__,
    "scipy": getattr(__import__("scipy"), "__version__", "?"),
    "pymc": pm.__version__,
    "arviz": az.__version__,
    "xgboost": xgb.__version__,
    "optuna": optuna.__version__,
    "matplotlib": matplotlib.__version__,
    "seaborn": sns.__version__,
}
_col_w = max(len(k) for k in _DEPS) + 2
print("Dependency versions")
print("─" * (_col_w + 12))
for _lib, _ver in _DEPS.items():
    print(f"  {_lib:<{_col_w}}{_ver}")
print("─" * (_col_w + 12))


### 1.2 `DataConfig`

Controls where raw match data is loaded from and which time window is included.

In [ ]:
@dataclass(frozen=True)
class DataConfig:
    """Configuration for raw data acquisition.

    Args:
        source_url: Direct URL to the raw results CSV file.
        cache_path: Local path for caching the downloaded CSV. ``None``
            disables caching and forces a fresh download on every run.
        start_date: Earliest match date to include in the analysis.
            Matches played before this date are discarded after ingestion.
    """

    source_url: str
    cache_path: pathlib.Path | None
    start_date: datetime.date


### 1.3 `TeamFilterConfig`

Controls which teams are included in the model universe. Teams below the `min_matches` threshold are excluded from fitting and from evaluation rows that involve them. The exclusion count is always reported, never silently dropped.

In [ ]:
@dataclass(frozen=True)
class TeamFilterConfig:
    """Configuration for the core-team universe.

    Args:
        min_matches: Minimum number of matches (home + away combined) a team
            must have played in the *training window* to be included in model
            fitting and backtest evaluation.  Teams below this threshold are
            excluded silently only after the exclusion count has been printed.
    """

    min_matches: int


### 1.4 `WeightingConfig`

Controls both sources of per-match weight:

- **Time decay** — exponential half-life; a match `half_life_days` before the reference date gets half the weight of a match on the reference date.
- **Tournament importance** — exact-string lookup against `tournament_weight_table`. Unrecognized names fall back to `default_tournament_weight`.

> **Implementation note:** the weight table must be keyed on the *exact* tournament name strings present in the dataset. Substring or accent-normalized matching is forbidden — two confirmed bugs in an earlier draft arose from precisely this pattern.

In [ ]:
def get_tournament_weight_table() -> dict[str, float]:
    """Return the canonical tournament-importance weight table.

    Keys are **exact** strings as they appear in the ``results.csv``
    ``tournament`` column (including accented characters). Weights are
    multiplicative scalars applied before the time-decay weight.

    Returns:
        Mapping from tournament name to importance weight.

    Note:
        An exact-string match is intentional: substring or
        accent-normalised matching introduced two confirmed bugs in an
        earlier draft (silent Copa America misweighting and World Cup
        qualifier/final conflation).
    """
    return {
        # Tier 1 - World Cup finals
        "FIFA World Cup": 4.0,
        # Tier 2 - Continental finals
        "Copa América": 3.5,
        "UEFA Euro": 3.5,
        "African Cup of Nations": 3.0,
        "AFC Asian Cup": 3.0,
        "CONCACAF Gold Cup": 2.5,
        # Tier 3 - Continental qualifying
        "FIFA World Cup qualification": 2.5,
        "FIFA World Cup qualification - CONMEBOL": 2.5,
        "FIFA World Cup qualification - UEFA": 2.5,
        "FIFA World Cup qualification - AFC": 2.5,
        "FIFA World Cup qualification - CAF": 2.5,
        "FIFA World Cup qualification - CONCACAF": 2.5,
        "FIFA World Cup qualification - OFC": 2.5,
        "UEFA Euro qualification": 2.0,
        "AFC Asian Cup qualification": 2.0,
        "Africa Cup of Nations qualification": 2.0,
        "African Cup of Nations qualification": 2.0,
        "CONCACAF Gold Cup qualification": 1.8,
        # Tier 4 - Nations Leagues
        "UEFA Nations League": 2.0,
        "CONMEBOL-UEFA Cup of Champions": 2.5,
        # Tier 5 - Friendly / uncapped
        "Friendly": 0.5,
        "FIFA World Cup qualification (play-off)": 3.0,
    }


@dataclass(frozen=True)
class WeightingConfig:
    """Configuration for per-match sample weighting.

    Args:
        half_life_days: Number of days before the reference date at which
            a match receives exactly half the base weight.  Exponential
            decay: ``weight = exp(-log(2) / half_life_days * delta_days)``.
        tournament_weight_table: Exact-string mapping from tournament name
            to a multiplicative importance scalar.  Keys must match the
            ``tournament`` column strings in the dataset verbatim.
        default_tournament_weight: Fallback weight for tournaments not
            found in ``tournament_weight_table``.  A warning is emitted
            for every unrecognised tournament name.
    """

    half_life_days: int
    tournament_weight_table: dict[str, float]
    default_tournament_weight: float

    def __post_init__(self) -> None:
        _required = {
            "Copa América", "FIFA World Cup", "UEFA Euro",
            "African Cup of Nations", "FIFA World Cup qualification", "Friendly",
        }
        _missing = _required - self.tournament_weight_table.keys()
        assert not _missing, (
            f"WeightingConfig.tournament_weight_table is missing required keys: {_missing}"
        )
        assert self.half_life_days > 0, "half_life_days must be positive"
        assert self.default_tournament_weight > 0, "default_tournament_weight must be positive"


### 1.5 `SplitConfig`

Defines the single chronological train/test boundary. Matches strictly before `cutoff_date` form the training set; matches on or after it form the holdout. This field is intentionally a named config value, not a hardcoded literal, to allow walk-forward sensitivity analysis without editing function bodies.

In [ ]:
@dataclass(frozen=True)
class SplitConfig:
    """Chronological train/test split configuration.

    Args:
        cutoff_date: Matches with ``date < cutoff_date`` form the training
            set; matches with ``date >= cutoff_date`` form the holdout.
            Using a named config value (rather than a hardcoded literal)
            allows walk-forward sensitivity analysis without editing any
            function bodies.
    """

    cutoff_date: datetime.date


### 1.6 `DixonColesConfig`

Controls the SciPy L-BFGS-B optimizer that fits the Dixon-Coles MAP estimates. `prior_scale` acts as a regularizer — larger values give flatter priors and allow more extreme attack/defense estimates for sparse teams.

In [ ]:
@dataclass(frozen=True)
class DixonColesConfig:
    """Configuration for the Dixon-Coles MAP optimizer.

    Args:
        optimizer_method: SciPy minimization method.  Must be ``'L-BFGS-B'``
            to support box constraints and an analytic gradient.
        max_iterations: Maximum iterations passed to the L-BFGS-B solver.
        prior_scale: Standard deviation of the zero-mean Gaussian prior
            placed on each attack and defense parameter.  Acts as an L2
            regularizer; larger values allow more extreme estimates for
            data-sparse teams.
    """

    optimizer_method: str
    max_iterations: int
    prior_scale: float

    def __post_init__(self) -> None:
        assert self.optimizer_method == "L-BFGS-B", (
            f"optimizer_method must be 'L-BFGS-B', got {self.optimizer_method!r}"
        )
        assert self.max_iterations > 0, "max_iterations must be positive"
        assert self.prior_scale > 0, "prior_scale must be positive"


### 1.7 `BayesianConfig`

Controls the PyMC / NUTS hierarchical model. Every stochastic aspect (draws, chains, seed) is a named field. Convergence thresholds (`rhat_threshold`, `min_ess`) are also stored here so that the pass/fail report in Section 8 references the same values.

In [ ]:
@dataclass(frozen=True)
class BayesianConfig:
    """Configuration for the PyMC / NUTS hierarchical model.

    Args:
        draws: Number of post-tuning NUTS samples per chain.
        tune: Number of tuning (warm-up) steps per chain.
        chains: Number of independent Markov chains.
        target_accept: Target Metropolis-Hastings acceptance rate for the
            dual-averaging step-size adaptor (NUTS).
        attack_prior_sigma: Scale of the ZeroSumNormal prior on attack
            parameters.
        defense_prior_sigma: Scale of the ZeroSumNormal prior on defense
            parameters.
        home_advantage_prior_mu: Prior mean for the home-advantage term.
        home_advantage_prior_sigma: Prior standard deviation for the
            home-advantage term.
        random_seed: Seed passed to ``pm.sample`` for reproducibility.
        rhat_threshold: Maximum acceptable R-hat value for convergence.
            Typically 1.01 (strict) or 1.05 (lenient).
        min_ess: Minimum acceptable effective sample size for convergence.
    """

    draws: int
    tune: int
    chains: int
    target_accept: float
    attack_prior_sigma: float
    defense_prior_sigma: float
    home_advantage_prior_mu: float
    home_advantage_prior_sigma: float
    random_seed: int
    rhat_threshold: float
    min_ess: int

    def __post_init__(self) -> None:
        assert self.draws > 0, "draws must be positive"
        assert self.tune > 0, "tune must be positive"
        assert 1 <= self.chains <= 8, "chains must be in [1, 8]"
        assert 0 < self.target_accept < 1, "target_accept must be in (0, 1)"
        assert self.attack_prior_sigma > 0
        assert self.defense_prior_sigma > 0
        assert self.home_advantage_prior_sigma > 0
        assert 1.0 < self.rhat_threshold <= 1.1, "rhat_threshold must be in (1.0, 1.1]"
        assert self.min_ess > 0


### 1.8 `FeatureConfig`

Specifies which features the XGBoost layer receives. The canonical `feature_columns` list is the single source of truth used in both training and prediction — it must never be manually re-typed at the call site.

In [ ]:
_REQUIRED_FEATURE_COLUMNS: list[str] = [
    "home_attack_dc",
    "home_defense_dc",
    "away_attack_dc",
    "away_defense_dc",
    "home_form_goals_for",
    "home_form_goals_against",
    "home_form_points",
    "away_form_goals_for",
    "away_form_goals_against",
    "away_form_points",
    "is_neutral",
]


@dataclass(frozen=True)
class FeatureConfig:
    """Configuration for XGBoost feature engineering.

    Args:
        rolling_window: Number of preceding matches used to compute
            each team's rolling form statistics.
        feature_columns: Ordered list of column names fed to XGBoost.
            This is the **single source of truth** used in both training
            and at prediction time; it must never be manually re-typed
            at a call site.
    """

    rolling_window: int
    feature_columns: list[str]

    def __post_init__(self) -> None:
        assert self.rolling_window >= 3, "rolling_window must be at least 3"
        _missing = set(_REQUIRED_FEATURE_COLUMNS) - set(self.feature_columns)
        assert not _missing, (
            f"FeatureConfig.feature_columns is missing required features: {_missing}"
        )


### 1.9 `OptunaConfig`

Controls the Optuna hyperparameter search executed strictly within the training set via expanding-window cross-validation. `sampler_seed` ensures reproducibility; `min_train_fraction` prevents the smallest folds from having too little data.

In [ ]:
@dataclass(frozen=True)
class OptunaConfig:
    """Configuration for the Optuna hyperparameter search.

    Args:
        n_trials: Total number of Optuna trials.
        timeout_seconds: Wall-clock time limit in seconds.  ``None``
            means no time limit; the search runs for exactly ``n_trials``.
        cv_folds: Number of expanding-window folds used inside each trial.
        min_train_fraction: Folds whose training fraction is below this
            threshold are skipped to prevent very small training windows.
        sampler_seed: Seed for the TPE sampler, ensuring reproducible
            trial sequences.
    """

    n_trials: int
    timeout_seconds: int | None
    cv_folds: int
    min_train_fraction: float
    sampler_seed: int

    def __post_init__(self) -> None:
        assert self.n_trials > 0, "n_trials must be positive"
        assert self.cv_folds >= 2, "cv_folds must be at least 2"
        assert 0.0 < self.min_train_fraction < 1.0, (
            "min_train_fraction must be in (0, 1)"
        )
        if self.timeout_seconds is not None:
            assert self.timeout_seconds > 0, "timeout_seconds must be positive"


### 1.10 `XGBoostConfig`

Fixed hyperparameters (not tuned by Optuna) and the Optuna search-space bounds are both stored here so they are auditable in one place.

In [ ]:
_COMMON_SEED: int = 42  # single source of truth for all stochastic seeds


@dataclass(frozen=True)
class XGBoostConfig:
    """Configuration for XGBoost Poisson regressors.

    Args:
        fixed_params: Hyperparameters that are *not* tuned by Optuna.
            Must include ``objective='count:poisson'`` and
            ``random_state``.
        search_space: Optuna search bounds for tunable hyperparameters.
            Each value is a ``(low, high)`` tuple of the same dtype as
            the parameter (``float`` for continuous, ``int`` for discrete).
    """

    fixed_params: dict[str, float | int | str]
    search_space: dict[str, tuple[float, float] | tuple[int, int]]

    def __post_init__(self) -> None:
        assert self.fixed_params.get("objective") == "count:poisson", (
            "fixed_params must include objective='count:poisson'"
        )
        assert "random_state" in self.fixed_params, (
            "fixed_params must include random_state"
        )
        _required_space = {
            "n_estimators", "max_depth", "learning_rate",
            "subsample", "colsample_bytree", "reg_alpha", "reg_lambda",
        }
        _missing = _required_space - self.search_space.keys()
        assert not _missing, (
            f"XGBoostConfig.search_space is missing required keys: {_missing}"
        )


### 1.11 `EnsembleConfig`

Controls how the three scoreline matrices are combined. `weights` starts as `None` and is populated after the backtest by `compute_ensemble_weights`. `temperature` controls how sharply the softmax concentrates on the best-performing model.

In [ ]:
@dataclass(frozen=True)
class EnsembleConfig:
    """Configuration for the ensemble matrix combination.

    Args:
        temperature: Softmax temperature applied to negative log-loss
            values when deriving model weights from backtest performance.
            Lower temperature concentrates weight on the best model;
            higher temperature gives a more uniform blend.
        weights: Pre-computed model weights keyed by model name.  Starts
            as ``None`` and is populated by ``compute_ensemble_weights``
            after the backtest.  The frozen dataclass is replaced with
            an updated instance at that point.
    """

    temperature: float
    weights: dict[str, float] | None = field(default=None)

    def __post_init__(self) -> None:
        assert self.temperature > 0, "temperature must be positive"
        if self.weights is not None:
            _total = sum(self.weights.values())
            assert abs(_total - 1.0) < 1e-6, (
                f"EnsembleConfig.weights must sum to 1.0, got {_total}"
            )


### 1.12 `FixtureConfig`

The single match being predicted. Teams are identified by FIFA code here and resolved to dataset team names in Section 3 via the `team_registry` module.

In [ ]:
@dataclass(frozen=True)
class FixtureConfig:
    """The single match being predicted.

    Teams are stored as FIFA three-letter codes here and resolved to
    dataset team-name strings in Section 3 (Team Registry).

    Args:
        home_team_fifa_code: FIFA code of the nominally "home" team
            (used for feature assignment even at neutral venues).
        away_team_fifa_code: FIFA code of the "away" team.
        match_date: Date on which the match will be / was played.
        is_neutral_venue: ``True`` when the match is played at a
            neutral location (home-advantage term is zeroed out).
    """

    home_team_fifa_code: str
    away_team_fifa_code: str
    match_date: datetime.date
    is_neutral_venue: bool

    def __post_init__(self) -> None:
        assert len(self.home_team_fifa_code) == 3, (
            f"home_team_fifa_code must be 3 characters, got {self.home_team_fifa_code!r}"
        )
        assert len(self.away_team_fifa_code) == 3, (
            f"away_team_fifa_code must be 3 characters, got {self.away_team_fifa_code!r}"
        )
        assert self.home_team_fifa_code != self.away_team_fifa_code, (
            "home and away FIFA codes must differ"
        )


### 1.13 `EvaluationConfig`

Governs the backtest evaluation. `max_goals` caps the scoreline matrix dimension (scorelines beyond this bound are collapsed into the boundary bin during evaluation).

In [ ]:
_REQUIRED_METRICS: list[str] = [
    "exact_score_accuracy",
    "outcome_accuracy",
    "log_loss",
    "brier_score",
    "home_goals_mae",
    "away_goals_mae",
    "home_goals_rmse",
    "away_goals_rmse",
]


@dataclass(frozen=True)
class EvaluationConfig:
    """Configuration for backtest evaluation.

    Args:
        max_goals: Maximum number of goals per team represented in the
            scoreline probability matrix.  Scorelines beyond this bound
            are collapsed into the boundary bin during evaluation.
        metrics: List of metric names to compute.  Must include all
            entries in ``_REQUIRED_METRICS``.
    """

    max_goals: int
    metrics: list[str]

    def __post_init__(self) -> None:
        assert self.max_goals >= 5, "max_goals must be at least 5"
        _missing = set(_REQUIRED_METRICS) - set(self.metrics)
        assert not _missing, (
            f"EvaluationConfig.metrics is missing required entries: {_missing}"
        )


### 1.14 `PipelineConfig` — top-level assembly

Composes all twelve sub-configs into one frozen object. This is the only object passed to orchestration functions in subsequent sections.

In [ ]:
@dataclass(frozen=True)
class PipelineConfig:
    """Top-level frozen configuration for the entire prediction pipeline.

    Composed of twelve nested sub-config objects.  This is the **only**
    object passed to orchestration functions in subsequent sections.
    No global mutable state is used anywhere in the notebook.

    Args:
        data: Raw data acquisition settings.
        team_filter: Core-team universe selection.
        weighting: Per-match time-decay and tournament-importance weights.
        split: Chronological train/test boundary.
        dixon_coles: Dixon-Coles MAP optimizer settings.
        bayesian: PyMC / NUTS hierarchical model settings.
        features: XGBoost feature engineering settings.
        optuna: Hyperparameter search settings.
        xgboost: XGBoost model settings.
        ensemble: Scoreline matrix combination settings.
        fixture: The headline fixture to predict.
        evaluation: Backtest evaluation settings.
    """

    data: DataConfig
    team_filter: TeamFilterConfig
    weighting: WeightingConfig
    split: SplitConfig
    dixon_coles: DixonColesConfig
    bayesian: BayesianConfig
    features: FeatureConfig
    optuna: OptunaConfig
    xgboost: XGBoostConfig
    ensemble: EnsembleConfig
    fixture: FixtureConfig
    evaluation: EvaluationConfig


def build_default_pipeline_config() -> PipelineConfig:
    """Construct the default PipelineConfig with documented defaults.

    All tunable values are set here.  No magic numbers appear anywhere
    else in the notebook; modify this function to change pipeline behaviour.

    Returns:
        A fully validated, frozen PipelineConfig instance.
    """
    _weight_table = get_tournament_weight_table()

    return PipelineConfig(
        data=DataConfig(
            source_url=(
                "https://raw.githubusercontent.com/martj42/"
                "international_results/master/results.csv"
            ),
            cache_path=pathlib.Path("results_cache.csv"),
            start_date=datetime.date(2018, 1, 1),
        ),
        team_filter=TeamFilterConfig(min_matches=10),
        weighting=WeightingConfig(
            half_life_days=365 * 3,   # 3-year half-life
            tournament_weight_table=_weight_table,
            default_tournament_weight=1.0,
        ),
        split=SplitConfig(cutoff_date=datetime.date(2024, 1, 1)),
        dixon_coles=DixonColesConfig(
            optimizer_method="L-BFGS-B",
            max_iterations=2000,
            prior_scale=1.0,
        ),
        bayesian=BayesianConfig(
            draws=1000,
            tune=1000,
            chains=2,
            target_accept=0.9,
            attack_prior_sigma=0.5,
            defense_prior_sigma=0.5,
            home_advantage_prior_mu=0.1,
            home_advantage_prior_sigma=0.3,
            random_seed=_COMMON_SEED,
            rhat_threshold=1.05,
            min_ess=200,
        ),
        features=FeatureConfig(
            rolling_window=10,
            feature_columns=list(_REQUIRED_FEATURE_COLUMNS),
        ),
        optuna=OptunaConfig(
            n_trials=60,
            timeout_seconds=300,
            cv_folds=4,
            min_train_fraction=0.3,
            sampler_seed=_COMMON_SEED,
        ),
        xgboost=XGBoostConfig(
            fixed_params={
                "objective": "count:poisson",
                "random_state": _COMMON_SEED,
                "tree_method": "hist",
                "eval_metric": "poisson-nloglik",
            },
            search_space={
                "n_estimators": (100, 600),
                "max_depth": (3, 8),
                "learning_rate": (0.01, 0.3),
                "subsample": (0.6, 1.0),
                "colsample_bytree": (0.6, 1.0),
                "reg_alpha": (0.0, 5.0),
                "reg_lambda": (0.5, 5.0),
            },
        ),
        ensemble=EnsembleConfig(temperature=1.0),
        fixture=FixtureConfig(
            home_team_fifa_code="ARG",
            away_team_fifa_code="AUT",
            match_date=datetime.date(2026, 6, 22),
            is_neutral_venue=True,
        ),
        evaluation=EvaluationConfig(
            max_goals=10,
            metrics=list(_REQUIRED_METRICS),
        ),
    )


# ── Instantiate and display ──────────────────────────────────────────────────
CFG: PipelineConfig = build_default_pipeline_config()

_SEP = "─" * 60
print(_SEP)
print("PipelineConfig — default values")
print(_SEP)
_fields_map = {
    "data":         CFG.data,
    "team_filter":  CFG.team_filter,
    "weighting":    f"half_life_days={CFG.weighting.half_life_days}, "
                    f"default_weight={CFG.weighting.default_tournament_weight}",
    "split":        CFG.split,
    "dixon_coles":  CFG.dixon_coles,
    "bayesian":     f"draws={CFG.bayesian.draws}, tune={CFG.bayesian.tune}, "
                    f"chains={CFG.bayesian.chains}, seed={CFG.bayesian.random_seed}",
    "features":     f"rolling_window={CFG.features.rolling_window}, "
                    f"n_features={len(CFG.features.feature_columns)}",
    "optuna":       CFG.optuna,
    "ensemble":     CFG.ensemble,
    "fixture":      CFG.fixture,
    "evaluation":   CFG.evaluation,
}
for _name, _val in _fields_map.items():
    print(f"  {_name:<14}: {_val}")
print(_SEP)


---

## Section 2 — Data Ingestion

Downloads (or reads from cache) the `results.csv` file from the `martj42/international_results` repository, parses the `date` column to `pl.Date`, and validates that the expected schema is present before any downstream code runs.

**Leakage note:** this section is read-only. No filtering, splitting, or weighting happens here.

### 2.1 Schema constants

Declare the expected column names and dtypes as module-level constants so that `validate_schema` can reference them without magic strings.

In [ ]:
# Expected schema: column name -> Polars dtype
EXPECTED_COLUMNS: dict[str, type[pl.DataType]] = {
    "date":       pl.Date,
    "home_team":  pl.Utf8,
    "away_team":  pl.Utf8,
    "home_score": pl.Int64,
    "away_score": pl.Int64,
    "tournament": pl.Utf8,
    "city":       pl.Utf8,
    "country":    pl.Utf8,
    "neutral":    pl.Boolean,
}


### 2.2 `fetch_results_csv`

Fetches the CSV from `DataConfig.source_url`. If `cache_path` is set and the file already exists locally, it is read from disk instead. The `date` column is parsed to `pl.Date` during ingestion, and missing scores (future fixtures) are represented as nulls via `null_values=["NA"]`.

In [ ]:
def fetch_results_csv(
    source_url: str,
    cache_path: pathlib.Path | None,
) -> pl.DataFrame:
    """Fetch the international results CSV and return a typed Polars DataFrame.

    If ``cache_path`` is provided and the file already exists on disk, it
    is read from disk.  Otherwise the file is downloaded from
    ``source_url`` and, if ``cache_path`` is set, written to disk for
    future runs.

    The ``date`` column is parsed directly to ``pl.Date`` during
    ingestion.  Missing scores (unplayed fixtures) are represented as
    ``null`` via ``null_values=['NA']``.

    Args:
        source_url: Direct URL to the raw ``results.csv`` file.
        cache_path: Optional local path for caching.

    Returns:
        Typed Polars DataFrame with columns matching ``EXPECTED_COLUMNS``.

    Raises:
        RuntimeError: If the download fails or the resulting frame is empty.
    """
    if cache_path is not None and cache_path.exists():
        print(f"  ↳ Loading from cache: {cache_path}")
        raw_bytes = cache_path.read_bytes()
    else:
        import urllib.request
        print(f"  ↳ Downloading from: {source_url}")
        try:
            with urllib.request.urlopen(source_url, timeout=60) as resp:
                raw_bytes = resp.read()
        except Exception as exc:  # noqa: BLE001
            raise RuntimeError(f"Failed to download results CSV: {exc}") from exc

        if cache_path is not None:
            cache_path.write_bytes(raw_bytes)
            print(f"  ↳ Saved to cache: {cache_path}")

    import io
    df = pl.read_csv(
        io.BytesIO(raw_bytes),
        null_values=["NA"],
        schema_overrides={
            "home_score": pl.Int64,
            "away_score": pl.Int64,
            "neutral":    pl.Boolean,
        },
        try_parse_dates=False,
    )

    # Parse date column explicitly to pl.Date
    df = df.with_columns(pl.col("date").str.to_date("%Y-%m-%d"))

    if df.is_empty():
        raise RuntimeError("Fetched CSV resulted in an empty DataFrame.")

    return df


### 2.3 `validate_schema`

Asserts that every column in `EXPECTED_COLUMNS` is present and has the correct dtype. Raises a descriptive `ValueError` immediately if the upstream schema has changed, rather than letting a `KeyError` or `InvalidOperationError` surface deep inside the Dixon-Coles fit.

In [ ]:
def validate_schema(
    df: pl.DataFrame,
    expected: dict[str, type[pl.DataType]] | None = None,
    start_date: datetime.date | None = None,
    stale_days: int = 7,
) -> None:
    """Assert that the DataFrame matches the expected schema.

    Performs three checks:
    1. All expected column names are present (raises on diff).
    2. Each column's dtype matches the expected dtype (raises on mismatch).
    3. The most recent ``date`` value is not more than ``stale_days`` old
       (emits a warning if stale, does not raise).

    Args:
        df: The raw ingested DataFrame to validate.
        expected: Schema mapping.  Defaults to ``EXPECTED_COLUMNS``.
        start_date: If provided, asserts ``df['date'].min() <=
            start_date`` (i.e. data reaches back far enough).
        stale_days: Warn if the most recent date is older than this many
            days relative to today.

    Raises:
        ValueError: If any expected columns are missing or have wrong dtypes.
    """
    if expected is None:
        expected = EXPECTED_COLUMNS

    actual_cols = set(df.columns)
    expected_cols = set(expected.keys())

    # 1. Column presence
    missing = expected_cols - actual_cols
    if missing:
        raise ValueError(
            f"validate_schema: missing columns {sorted(missing)}. "
            f"Present columns: {sorted(actual_cols)}"
        )

    # 2. Dtype check
    mismatches: list[str] = []
    schema_map = {col: df.schema[col] for col in df.columns}
    for col, exp_dtype in expected.items():
        actual_dtype = schema_map[col]
        # Compare base dtype (ignoring nullability wrappers)
        if not isinstance(actual_dtype, exp_dtype):
            mismatches.append(
                f"  {col}: expected {exp_dtype.__name__}, got {actual_dtype}"
            )
    if mismatches:
        raise ValueError(
            "validate_schema: dtype mismatches:\n" + "\n".join(mismatches)
        )

    # 3. Staleness warning
    max_date: datetime.date = df["date"].max()
    today = datetime.date.today()
    delta_days = (today - max_date).days
    if delta_days > stale_days:
        warnings.warn(
            f"validate_schema: most recent match date is {max_date} "
            f"({delta_days} days ago). Dataset may be stale.",
            stacklevel=2,
        )

    # 4. Optional start-date coverage check
    if start_date is not None:
        min_date: datetime.date = df["date"].min()
        assert min_date <= start_date, (
            f"validate_schema: earliest date {min_date} is after "
            f"start_date {start_date}"
        )


### 2.4 Execute and inspect

Run the fetch and validation, then print a concise data quality summary: row count, date range, proportion of null scores (future fixtures), and the number of distinct teams and tournaments.

In [ ]:
print("Section 2 — Data Ingestion")
print("─" * 60)

RAW_DF: pl.DataFrame = fetch_results_csv(
    CFG.data.source_url,
    CFG.data.cache_path,
)
validate_schema(RAW_DF, start_date=CFG.data.start_date)

# ── Summary statistics ───────────────────────────────────────────────────────
_n_rows, _n_cols = RAW_DF.shape
_date_min: datetime.date = RAW_DF["date"].min()
_date_max: datetime.date = RAW_DF["date"].max()
_null_score_rows = RAW_DF.filter(
    pl.col("home_score").is_null() | pl.col("away_score").is_null()
).height
_n_teams = (
    pl.concat([RAW_DF["home_team"], RAW_DF["away_team"]])
    .n_unique()
)
_n_tournaments = RAW_DF["tournament"].n_unique()

print(f"  Rows:              {_n_rows:,}")
print(f"  Columns:           {_n_cols}")
print(f"  Date range:        {_date_min} → {_date_max}")
print(f"  Null-score rows:   {_null_score_rows:,}  (unplayed fixtures)")
print(f"  Distinct teams:    {_n_teams:,}")
print(f"  Distinct tourn.:   {_n_tournaments:,}")
print()


---

## Section 3 — Team Registry

Resolves FIFA three-letter codes to the exact team name strings used in `results.csv`, selects the core-team universe (teams with enough match history to support stable rating), and verifies that both headline fixture teams are in that universe.

**Why this matters:** the dataset uses full English names (`'Argentina'`, not `'ARG'`). The `FixtureConfig` stores FIFA codes (human-readable, standard) and this section is the single translation point.

### 3.1 FIFA code mapping

Load the `FIFA_TO_DATASET_TEAM` dictionary from `fifa_country_codes.py` and wrap it in a typed resolver function.

In [ ]:
from fifa_country_codes import FIFA_TO_DATASET_TEAM


def load_fifa_code_mapping(source: dict[str, str]) -> dict[str, str]:
    """Wrap a FIFA-code to dataset-team-name dictionary in a typed copy.

    Args:
        source: The canonical ``FIFA_TO_DATASET_TEAM`` dict (or any
            equivalent mapping).

    Returns:
        A new ``dict[str, str]`` containing the same key-value pairs.
    """
    return dict(source)


def resolve_team_name(fifa_code: str, mapping: dict[str, str]) -> str:
    """Resolve a FIFA three-letter code to the dataset team name string.

    Args:
        fifa_code: Upper-case three-letter FIFA code (e.g. ``'ARG'``).
        mapping: Code to team-name dictionary (from ``load_fifa_code_mapping``).

    Returns:
        The exact team name string used in ``results.csv``.

    Raises:
        KeyError: If ``fifa_code`` is not in ``mapping``, with a
            descriptive message listing the code.
    """
    try:
        return mapping[fifa_code]
    except KeyError:
        raise KeyError(
            f"FIFA code {fifa_code!r} not found in the code mapping. "
            f"Check fifa_country_codes.py for the correct code."
        ) from None


CODE_MAP: dict[str, str] = load_fifa_code_mapping(FIFA_TO_DATASET_TEAM)
print(f"FIFA code mapping loaded: {len(CODE_MAP):,} entries")


### 3.2 Fixture team resolution

Resolve the two FIFA codes in `FixtureConfig` to their dataset team-name strings and store them as module-level constants used throughout the notebook.

In [ ]:
FIXTURE_HOME_TEAM: str = resolve_team_name(CFG.fixture.home_team_fifa_code, CODE_MAP)
FIXTURE_AWAY_TEAM: str = resolve_team_name(CFG.fixture.away_team_fifa_code, CODE_MAP)

print(f"  Home team ({CFG.fixture.home_team_fifa_code}): {FIXTURE_HOME_TEAM}")
print(f"  Away team ({CFG.fixture.away_team_fifa_code}): {FIXTURE_AWAY_TEAM}")
print(f"  Match date:           {CFG.fixture.match_date}")
print(f"  Neutral venue:        {CFG.fixture.is_neutral_venue}")

# Verify both team names appear in the raw data
_all_teams = set(RAW_DF["home_team"].to_list()) | set(RAW_DF["away_team"].to_list())
assert FIXTURE_HOME_TEAM in _all_teams, (
    f"{FIXTURE_HOME_TEAM!r} not found in RAW_DF — check FIFA code mapping"
)
assert FIXTURE_AWAY_TEAM in _all_teams, (
    f"{FIXTURE_AWAY_TEAM!r} not found in RAW_DF — check FIFA code mapping"
)


### 3.3 Core-team selection

Identify teams with at least `TeamFilterConfig.min_matches` appearances (home + away combined) in the training window. Teams below this threshold are excluded from all model fits and from backtest evaluation rows that involve them. The exclusion count is printed, not silently swallowed.

In [ ]:
def select_core_teams(df: pl.DataFrame, min_matches: int) -> list[str]:
    """Return teams with at least min_matches appearances in df.

    Appearance counts are the sum of home and away occurrences combined.
    This function must be called on the **training** frame only to
    prevent test-set match frequencies from influencing which teams
    are in the model universe.

    Args:
        df: Match DataFrame (training window only).
        min_matches: Minimum number of matches required for inclusion.

    Returns:
        Sorted list of team name strings meeting the threshold.
    """
    home_counts = (
        df.group_by("home_team")
        .agg(pl.len().alias("n"))
        .rename({"home_team": "team"})
    )
    away_counts = (
        df.group_by("away_team")
        .agg(pl.len().alias("n"))
        .rename({"away_team": "team"})
    )
    total_counts = (
        pl.concat([home_counts, away_counts])
        .group_by("team")
        .agg(pl.col("n").sum())
    )
    core = (
        total_counts
        .filter(pl.col("n") >= min_matches)
        ["team"]
        .to_list()
    )
    return sorted(core)


def filter_to_core_teams(df: pl.DataFrame, core_teams: list[str]) -> pl.DataFrame:
    """Filter a match DataFrame to rows where both teams are in core_teams.

    Args:
        df: Match DataFrame (played matches only).
        core_teams: List of team name strings in the model universe.

    Returns:
        Filtered DataFrame; the original row order is preserved.
    """
    _core_set = set(core_teams)
    return df.filter(
        pl.col("home_team").is_in(_core_set)
        & pl.col("away_team").is_in(_core_set)
    )


print("Core-team selection functions defined.")
print("(Actual call deferred to Section 4 — requires TRAIN_RAW)")


---

## Section 4 — Data Preparation

Applies the sequence of filters and splits that produces the four frames all subsequent sections operate on:

- `PLAYED_DF` — all completed matches from `start_date` onward
- `UPCOMING_DF` — fixtures with no recorded score (includes the headline fixture)
- `TRAIN_DF` — played matches strictly before `cutoff_date`, filtered to core teams
- `TEST_DF` — played matches from `cutoff_date` onward, filtered to core teams

**Order matters:** (1) split played/upcoming on the raw frame, (2) apply date filter, (3) derive core-team universe from the *train* portion only, (4) filter both train and test to that universe. This order prevents information from test or future matches from influencing which teams are included.

### 4.1 Played / upcoming split

Separate rows with recorded scores from those without. Nulls in `home_score` / `away_score` indicate an unplayed fixture.

In [ ]:
def split_played_and_upcoming(
    df: pl.DataFrame,
) -> tuple[pl.DataFrame, pl.DataFrame]:
    """Split a match DataFrame into played and upcoming fixtures.

    Played matches have non-null scores in *both* ``home_score`` and
    ``away_score``.  Fixtures with at least one null score are treated
    as upcoming (i.e. not yet played).

    Args:
        df: Raw match DataFrame (all rows, including future fixtures).

    Returns:
        Tuple of (played_df, upcoming_df).  Their row counts sum to
        ``len(df)``.

    Raises:
        AssertionError: If the union of row counts does not equal
            ``len(df)``.
    """
    played = df.filter(
        pl.col("home_score").is_not_null() & pl.col("away_score").is_not_null()
    )
    upcoming = df.filter(
        pl.col("home_score").is_null() | pl.col("away_score").is_null()
    )
    assert played.height + upcoming.height == df.height, (
        "split_played_and_upcoming: played + upcoming != total rows. "
        "Check for unexpected null patterns."
    )
    return played, upcoming


### 4.2 Date filter

Restrict played matches to `[DataConfig.start_date, ∞)`. Attach a sequential `match_id` index (used by the rolling-form logic in Section 7) before any subsequent operations.

In [ ]:
def filter_from_date(
    df: pl.DataFrame,
    start_date: datetime.date,
) -> pl.DataFrame:
    """Restrict a match DataFrame to rows on or after start_date.

    A sequential ``match_id`` index column is attached after filtering
    so that rolling-form logic in Section 7 can use it as a stable,
    monotonically increasing row identifier.

    Args:
        df: Played match DataFrame (scores must be non-null).
        start_date: Earliest date to include (inclusive).

    Returns:
        Filtered DataFrame with an additional ``match_id`` column
        (``UInt32``) starting at 0.
    """
    filtered = df.filter(pl.col("date") >= start_date)
    filtered = filtered.with_row_index("match_id")
    return filtered


### 4.3 Chronological train / test split

Split by date, not by row index or random shuffle. Every match in `TEST_DF` must have been played after every match in `TRAIN_DF`.

In [ ]:
def train_test_split_by_date(
    df: pl.DataFrame,
    cutoff_date: datetime.date,
) -> tuple[pl.DataFrame, pl.DataFrame]:
    """Split a played-match DataFrame at a chronological boundary.

    Every match in the returned test set has ``date >= cutoff_date``;
    every match in the training set has ``date < cutoff_date``.
    No row appears in both sets.

    Args:
        df: Played match DataFrame, sorted by date (ascending).
        cutoff_date: Exclusive upper bound for the training set.

    Returns:
        Tuple of (train_df, test_df).

    Raises:
        AssertionError: If the maximum training date is not strictly
            before ``cutoff_date``, or if the minimum test date is
            not ``>= cutoff_date``.
    """
    train = df.filter(pl.col("date") < cutoff_date)
    test  = df.filter(pl.col("date") >= cutoff_date)

    if train.height > 0 and test.height > 0:
        _train_max: datetime.date = train["date"].max()
        _test_min:  datetime.date = test["date"].min()
        assert _train_max < cutoff_date, (
            f"train_test_split_by_date: max train date {_train_max} "
            f">= cutoff_date {cutoff_date}"
        )
        assert _test_min >= cutoff_date, (
            f"train_test_split_by_date: min test date {_test_min} "
            f"< cutoff_date {cutoff_date}"
        )
    return train, test


### 4.4 Core-team universe derivation and filtering

The core-team set is derived from `TRAIN_RAW` only, then applied as a filter to both `TRAIN_RAW` and `TEST_RAW`. Deriving it from the full played history would let information about test-period match frequencies influence which teams are included.

In [ ]:
print("Section 4 — Data Preparation")
print("─" * 60)

# 4.1 Played / upcoming split
PLAYED_DF, UPCOMING_DF = split_played_and_upcoming(RAW_DF)
print(f"  Played rows:    {PLAYED_DF.height:,}")
print(f"  Upcoming rows:  {UPCOMING_DF.height:,}")

# 4.2 Date filter + match_id
_rows_before = PLAYED_DF.height
PLAYED_FILTERED_DF: pl.DataFrame = filter_from_date(PLAYED_DF, CFG.data.start_date)
_rows_after = PLAYED_FILTERED_DF.height
print(f"  Rows after date filter (>= {CFG.data.start_date}): "
      f"{_rows_after:,}  (dropped {_rows_before - _rows_after:,})")

# 4.3 Chronological train / test split (raw, pre-core-team filter)
TRAIN_RAW, TEST_RAW = train_test_split_by_date(
    PLAYED_FILTERED_DF, CFG.split.cutoff_date
)
print(f"  TRAIN_RAW rows: {TRAIN_RAW.height:,}  "
      f"(up to {TRAIN_RAW['date'].max()})")
print(f"  TEST_RAW rows:  {TEST_RAW.height:,}  "
      f"(from {TEST_RAW['date'].min()})")

# 4.4a Core-team universe (derived from TRAIN_RAW only — no test leakage)
CORE_TEAMS: list[str] = select_core_teams(TRAIN_RAW, CFG.team_filter.min_matches)

_n_all_teams = (
    pl.concat([TRAIN_RAW["home_team"], TRAIN_RAW["away_team"]]).n_unique()
)
print(f"\n  Teams in training data:  {_n_all_teams:,}")
print(f"  Core teams (>= {CFG.team_filter.min_matches} matches): {len(CORE_TEAMS):,}")
print(f"  Teams excluded:          {_n_all_teams - len(CORE_TEAMS):,}")

# 4.4b Filter to core teams
TRAIN_DF: pl.DataFrame = filter_to_core_teams(TRAIN_RAW, CORE_TEAMS)
TEST_DF:  pl.DataFrame = filter_to_core_teams(TEST_RAW,  CORE_TEAMS)

# Full played history restricted to core teams (used for production refit)
CORE_MATCH_DF: pl.DataFrame = filter_to_core_teams(PLAYED_FILTERED_DF, CORE_TEAMS)

_test_dropped = TEST_RAW.height - TEST_DF.height
print(f"\n  TRAIN_DF rows (core teams): {TRAIN_DF.height:,}")
print(f"  TEST_DF rows (core teams):  {TEST_DF.height:,}  "
      f"(dropped {_test_dropped:,} rows with non-core teams)")
print(f"  CORE_MATCH_DF rows:         {CORE_MATCH_DF.height:,}")

# 4.4c Assert fixture teams are in core universe
assert FIXTURE_HOME_TEAM in CORE_TEAMS, (
    f"Fixture home team {FIXTURE_HOME_TEAM!r} not in CORE_TEAMS. "
    f"Reduce TeamFilterConfig.min_matches."
)
assert FIXTURE_AWAY_TEAM in CORE_TEAMS, (
    f"Fixture away team {FIXTURE_AWAY_TEAM!r} not in CORE_TEAMS. "
    f"Reduce TeamFilterConfig.min_matches."
)

# 4.4d Team index: stable mapping from team name to integer index
TEAM_INDEX: dict[str, int] = {team: i for i, team in enumerate(CORE_TEAMS)}

print(f"\n  TEAM_INDEX size:            {len(TEAM_INDEX):,}")
print(f"  {FIXTURE_HOME_TEAM} index:  {TEAM_INDEX[FIXTURE_HOME_TEAM]}")
print(f"  {FIXTURE_AWAY_TEAM} index:  {TEAM_INDEX[FIXTURE_AWAY_TEAM]}")


---

## Section 5 — Match Weighting

Attaches per-match weights combining two independent signals:

1. **Time-decay weight** — exponential decay so that older matches contribute less. The reference date is the *training-set maximum date* for the backtest pass, and *today* for the production pass. This distinction matters: using today as the reference in the backtest pass would over-weight the most recent train matches relative to the actual forecasting horizon.

2. **Tournament-importance weight** — exact-string lookup. Higher weights for major tournaments and their qualifiers than for friendlies.

### 5.1 Time-decay weights

Exponential half-life: `weight = exp(-log(2) / half_life_days * days_before_reference)`.

In [ ]:
# TODO: Implement compute_time_decay_weight(dates: pl.Series, reference_date: datetime.date, half_life_days: int) -> pl.Series
# TODO: Compute (reference_date - date).total_days() then apply the exponential formula
# TODO: Verify that a match on the reference_date gets weight 1.0 and a match half_life_days before gets weight 0.5

### 5.2 Tournament-importance weights

Exact-key lookup against `WeightingConfig.tournament_weight_table`. Unknown tournaments fall back to `default_tournament_weight`. A warning is emitted (not silently suppressed) for every unrecognized tournament name, listing the name and how many matches it affects.

In [ ]:
# TODO: Implement compute_tournament_weight(tournaments: pl.Series, config: WeightingConfig) -> pl.Series
# TODO: Use exact-string dict lookup — no substring matching, no accent normalization
# TODO: For each unrecognized tournament, emit a warning with the name and match count
# TODO: Spot-check: verify 'Copa América' maps to the intended weight (requires the accent to be preserved in the key)

### 5.3 Combined match weights

The final `match_weight` column is the product of time-decay and tournament-importance weights, then normalized so the mean weight equals 1.0 (prevents the loss scale from drifting as window sizes change).

In [ ]:
# TODO: Implement compute_match_weights(df: pl.DataFrame, reference_date: datetime.date, config: WeightingConfig) -> pl.DataFrame
# TODO: Attach time_weight, tournament_weight, and match_weight (= time_weight * tournament_weight, then mean-normalized) columns
# TODO: Return the enriched frame

### 5.4 Compute for backtest and production

Weights are computed twice with different reference dates:

- **Backtest weights** use `TRAIN_DF['date'].max()` as the reference date (used in Sections 6, 8, 9, 10).
- **Production weights** use today's date as the reference date (used in Section 14).

In [ ]:
# TODO: TRAIN_REFERENCE_DATE = TRAIN_DF['date'].max()
# TODO: TRAIN_WEIGHTED_DF = compute_match_weights(TRAIN_DF, TRAIN_REFERENCE_DATE, CFG.weighting)
# TODO: Print weight summary: min, max, mean, and the top-5 and bottom-5 weighted matches
# TODO: Store PRODUCTION_REFERENCE_DATE = datetime.date.today()  — used in Section 14

---

## Section 6 — Dixon-Coles Fit (Backtest)

Fits the Dixon-Coles model via weighted maximum-a-posteriori (MAP) estimation using SciPy's L-BFGS-B optimizer. The model estimates per-team attack and defense parameters, a global home-advantage term, a global intercept (log baseline scoring rate), and a `rho` parameter for the low-score correlation correction.

**Identifiability:** attack and defense vectors are constrained to sum to zero (the last team's value is derived as the negative sum of the others), which prevents a degenerate family of equivalent solutions.

**This section fits on `TRAIN_WEIGHTED_DF` only.** The resulting `DixonColesRatings` object is used both as a standalone estimator (Section 11) and as the source of static features for XGBoost (Section 7).

### 6.1 Dataclass: `DixonColesRatings`

Stores the complete MAP solution as named fields so callers never index into raw arrays.

In [ ]:
# TODO: Define DixonColesRatings dataclass: teams: list[str], attack: dict[str, float], defense: dict[str, float], home_advantage: float, intercept: float, rho: float

### 6.2 Parameter packing / unpacking

The optimizer works on a flat 1-D array. The packing convention must be documented and tested: `[attack[0..n-2], defense[0..n-2], home_advantage, intercept, rho_raw]`, where `rho = 0.2 * tanh(rho_raw)` to keep `rho` in `(-0.2, 0.2)` without box constraints.

In [ ]:
# TODO: Implement _pack_params(ratings: DixonColesRatings) -> np.ndarray
# TODO: Implement _unpack_params(params: np.ndarray, n_teams: int) -> tuple[np.ndarray, np.ndarray, float, float, float]
# TODO: Unit test: assert _unpack_params(_pack_params(some_ratings), n) reproduces the original values

### 6.3 Dixon-Coles `tau` correction

Applies the Dixon-Coles low-score multiplicative adjustment to the joint Poisson log-likelihood for scorelines (0,0), (1,0), (0,1), and (1,1). All other scorelines have `tau = 1.0`.

In [ ]:
# TODO: Implement dixon_coles_tau(home_goals: np.ndarray, away_goals: np.ndarray, lam: np.ndarray, mu: np.ndarray, rho: float) -> np.ndarray — vectorized, returns tau per match
# TODO: Clip tau to a small positive floor before taking log to guard against edge-case rho values

### 6.4 Weighted negative log-posterior and analytic gradient

The objective is the weighted sum of per-match log-likelihoods (Poisson + tau) plus a Gaussian log-prior on all parameters (regularization, scale = `prior_scale`). Providing the analytic gradient to L-BFGS-B cuts optimizer iterations by a factor of ~10.

In [ ]:
# TODO: Implement negative_log_posterior(params: np.ndarray, home_idx: np.ndarray, away_idx: np.ndarray, home_goals: np.ndarray, away_goals: np.ndarray, is_neutral: np.ndarray, weights: np.ndarray, n_teams: int, config: DixonColesConfig) -> float
# TODO: Implement negative_log_posterior_gradient(params: np.ndarray, ...) -> np.ndarray — same signature

### 6.5 Gradient verification

Before trusting the analytic gradient in production, verify it against SciPy's `check_grad` on a small synthetic dataset. The error must be below `1e-4`.

In [ ]:
# TODO: Generate synthetic match data (small: ~8 teams, ~60 matches) with a known random seed
# TODO: Run scipy.optimize.check_grad(negative_log_posterior, negative_log_posterior_gradient, x0, *args)
# TODO: Assert error < 1e-4; print the result regardless

### 6.6 `fit_dixon_coles`

Top-level fitting function. Initialises the parameter vector, calls L-BFGS-B with the analytic gradient, checks optimizer convergence, and unpacks the result into a `DixonColesRatings` object.

In [ ]:
# TODO: Implement fit_dixon_coles(matches: pl.DataFrame, teams: list[str], config: DixonColesConfig) -> DixonColesRatings
# TODO: Initial parameter vector: zeros for attack/defense, 0.25 for home_advantage, 0.0 for intercept, 0.0 for rho_raw
# TODO: Assert optimizer convergence (warnflag == 0); print a warning but do not raise if it does not converge

### 6.7 Fit and inspect

Execute the fit and print a quick sanity-check: globally fitted scalar parameters, top-5 attack and top-5 defense teams, and the specific ratings for the headline fixture teams.

In [ ]:
# TODO: DC_RATINGS_BACKTEST: DixonColesRatings = fit_dixon_coles(TRAIN_WEIGHTED_DF, sorted(CORE_TEAMS), CFG.dixon_coles)
# TODO: Print home_advantage, intercept, rho
# TODO: Print top-5 attack and top-5 defense (lowest defense = best) teams
# TODO: Print attack and defense for FIXTURE_HOME_TEAM and FIXTURE_AWAY_TEAM

---

## Section 7 — Feature Engineering

Builds the feature table consumed by XGBoost (Section 10). Two feature groups are combined:

1. **Static Dixon-Coles features** — `home_attack_dc`, `home_defense_dc`, `away_attack_dc`, `away_defense_dc` — derived from `DC_RATINGS_BACKTEST` (train-only) and joined onto both train and test rows.

2. **Leak-free rolling form features** — `{home|away}_form_goals_{for|against}` and `{home|away}_form_points` — computed over the full chronological history (`CORE_MATCH_DF`, train+test together) using a `shift(1)` before the rolling window so each row only sees its own past, then re-split by date.

**Critical leakage rule:** `add_rolling_form_features` must be called on the **combined** `CORE_MATCH_DF` (not separately on train and test), otherwise test rows that happen to be a team's first post-cutoff match would have no rolling history at all. The function itself is leak-free by construction (shift before rolling); the split is re-applied afterward.

### 7.1 `add_rolling_form_features`

Pivots the match frame into a long team-perspective frame, computes `shift(1)` then `rolling_mean(window)` per team over `date`-sorted rows, then pivots back and joins home and away form columns onto the original frame.

In [ ]:
# TODO: Implement add_rolling_form_features(df: pl.DataFrame, window: int) -> pl.DataFrame
# TODO: Pivot to long format: one row per (match_id, team) perspective with goals_for, goals_against, points
# TODO: Sort by ['team', 'date', 'match_id'] before applying shift and rolling to ensure chronological order
# TODO: Shift each per-team series by 1 before rolling_mean to exclude the current match from its own feature
# TODO: Join home-perspective and away-perspective form columns back onto df using match_id
# TODO: Return df with six new columns: home_form_goals_for, home_form_goals_against, home_form_points, away_form_goals_for, away_form_goals_against, away_form_points

### 7.2 Leakage spot-check

Manually compute the expected rolling mean for one team over a known window and assert it matches the column value. This test must pass before proceeding.

In [ ]:
# TODO: Select a team that appears frequently (e.g. FIXTURE_HOME_TEAM)
# TODO: Extract their last 12 matches from CORE_MATCH_DF in chronological order
# TODO: Manually compute the expected form_goals_for for the final match as the mean of the preceding 10 goals_for values
# TODO: Assert the value matches FORM_DF's home_form_goals_for (or away_form_goals_for) for that match row
# TODO: Print pass/fail result

### 7.3 `add_dixon_coles_features`

Joins the four DC rating columns onto a match frame via an exact-match `replace_strict` on the team name columns. Unrated teams (those not in `DC_RATINGS_BACKTEST.attack`) receive a `null` rather than a silent 0.0 imputation.

In [ ]:
# TODO: Implement add_dixon_coles_features(df: pl.DataFrame, ratings: DixonColesRatings) -> pl.DataFrame
# TODO: Add: home_attack_dc, home_defense_dc, away_attack_dc, away_defense_dc using replace_strict with default=null
# TODO: Add: is_neutral as Int8 cast from the boolean neutral column

### 7.4 Assemble and re-split

Apply both feature functions to `CORE_MATCH_DF`, then re-split the result by date to obtain `TRAIN_FEATURES` and `TEST_FEATURES`. Join `TRAIN_WEIGHTED_DF`'s `match_weight` column onto `TRAIN_FEATURES` via `match_id`.

In [ ]:
# TODO: FORM_DF = add_rolling_form_features(CORE_MATCH_DF, CFG.features.rolling_window)
# TODO: FEATURE_DF = add_dixon_coles_features(FORM_DF, DC_RATINGS_BACKTEST)
# TODO: TRAIN_FEATURES = FEATURE_DF.filter(pl.col('date') < CFG.split.cutoff_date)
# TODO: TEST_FEATURES = FEATURE_DF.filter(pl.col('date') >= CFG.split.cutoff_date)
# TODO: Join match_weight from TRAIN_WEIGHTED_DF onto TRAIN_FEATURES on match_id
# TODO: Assert no nulls in CFG.features.feature_columns for TRAIN_FEATURES; report null count for TEST_FEATURES
# TODO: Print TRAIN_FEATURES and TEST_FEATURES shapes and null counts per feature column

### 7.5 `compute_current_form` (for production use)

An **unshifted** form snapshot giving each team's mean stats over their most recent `window` played matches. This is used only in Section 15 to build the feature row for the headline fixture — never for backtest training or test rows.

In [ ]:
# TODO: Implement compute_current_form(df: pl.DataFrame, window: int) -> pl.DataFrame
# TODO: Return one row per team with form_goals_for, form_goals_against, form_points (no shift — using all recent matches)
# TODO: Do NOT call this function here; it is deferred to Section 15 where the production models are available

---

## Section 8 — Bayesian Hierarchical Model (Backtest)

Builds and fits a Bayesian hierarchical Poisson model using PyMC / NUTS. The generative structure mirrors Dixon-Coles — log goal rates are a sum of a global intercept, a home-advantage term (zeroed for neutral-venue matches), and per-team attack and defense effects — but the Bayesian model produces a full posterior over every parameter rather than point estimates.

**Key design choices:**

- `ZeroSumNormal` priors enforce the sum-to-zero identifiability constraint and sample more efficiently than manual reparameterization.
- Per-observation likelihood contributions are scaled by `match_weight` via `pm.Potential` (PyMC distributions do not natively accept per-observation weights).
- Convergence is evaluated automatically via a `ConvergenceReport` dataclass; the notebook prints a pass/fail summary, not raw numbers for the reader to interpret.

**This section fits on `TRAIN_WEIGHTED_DF` only.**

### 8.1 Dataclasses: `ConvergenceReport` and `BayesianPosterior`

In [ ]:
# TODO: Define ConvergenceReport dataclass: max_rhat: float, min_ess: float, n_divergences: int, n_chains: int, passed: bool
# TODO: Define BayesianPosterior dataclass: teams: list[str], team_index: dict[str, int], attack_mean: dict[str, float], attack_sd: dict[str, float], defense_mean: dict[str, float], defense_sd: dict[str, float], home_advantage_mean: float, intercept_mean: float, convergence: ConvergenceReport

### 8.2 `build_bayesian_model`

Constructs the PyMC model object. The model is not sampled here; only the graph is built so that `fit_bayesian_model` can be called separately (and so the model can be inspected or modified before sampling).

In [ ]:
# TODO: Implement build_bayesian_model(home_idx: np.ndarray, away_idx: np.ndarray, home_goals: np.ndarray, away_goals: np.ndarray, is_neutral: np.ndarray, weights: np.ndarray, n_teams: int, config: BayesianConfig) -> pm.Model
# TODO: Priors: attack ~ ZeroSumNormal(sigma=config.attack_prior_sigma, shape=n_teams)
# TODO: Priors: defense ~ ZeroSumNormal(sigma=config.defense_prior_sigma, shape=n_teams)
# TODO: Priors: home_advantage ~ Normal(mu=config.home_advantage_prior_mu, sigma=config.home_advantage_prior_sigma)
# TODO: Priors: intercept ~ Normal(mu=0, sigma=0.5)
# TODO: home_edge_mask: 1.0 for non-neutral, 0.0 for neutral matches
# TODO: log_lambda = intercept + home_advantage * home_edge_mask + attack[home_idx] - defense[away_idx]
# TODO: log_mu = intercept + attack[away_idx] - defense[home_idx]
# TODO: Weighted likelihood via pm.Potential: sum(weights * pm.logp(Poisson(exp(log_lambda)), home_goals))
# TODO: Weighted likelihood via pm.Potential: sum(weights * pm.logp(Poisson(exp(log_mu)), away_goals))
# TODO: Add pm.Deterministic nodes for lambda_obs and mu_obs (needed for posterior-predictive checks)

### 8.3 `fit_bayesian_model`

Runs NUTS sampling with the parameters specified in `BayesianConfig`. Always runs all chains sequentially on `cores=1` to ensure consistent behaviour in constrained compute environments.

In [ ]:
# TODO: Implement fit_bayesian_model(model: pm.Model, config: BayesianConfig) -> az.InferenceData
# TODO: pm.sample(draws=config.draws, tune=config.tune, chains=config.chains, cores=1, target_accept=config.target_accept, random_seed=config.random_seed, progressbar=True)

### 8.4 `check_convergence`

Computes max R-hat, min ESS, and divergence count across the key parameters and returns a `ConvergenceReport`. The report's `passed` field is the single authoritative answer to 'did this model converge?'

In [ ]:
# TODO: Implement check_convergence(idata: az.InferenceData, config: BayesianConfig) -> ConvergenceReport
# TODO: Compute max R-hat over attack, defense, home_advantage, intercept via az.rhat()
# TODO: Compute min ESS over the same variables via az.ess()
# TODO: Count divergences from idata.sample_stats['diverging'].sum()
# TODO: passed = (max_rhat < config.rhat_threshold) and (min_ess >= config.min_ess) and (n_divergences == 0)

### 8.5 Posterior predictive checks

Simulate goal totals from the fitted model and compare to the observed training-set distribution. A well-fitting model should reproduce the observed mean goals per match and the proportion of 0-0 draws within a reasonable tolerance.

In [ ]:
# TODO: Implement posterior_predictive_check(idata: az.InferenceData, observed_home: np.ndarray, observed_away: np.ndarray) -> dict[str, float]
# TODO: Sample from the posterior predictive distribution using pm.sample_posterior_predictive inside the model context
# TODO: Return dict with: observed_mean_home, predicted_mean_home, observed_prop_00, predicted_prop_00, and similar for away
# TODO: Print PPC summary

### 8.6 `extract_posterior_means`

Collapses the full posterior into posterior means and standard deviations per team, packaged into a `BayesianPosterior` for use in Section 11 and Section 14.

In [ ]:
# TODO: Implement extract_posterior_means(idata: az.InferenceData, teams: list[str]) -> BayesianPosterior
# TODO: Average over chain and draw dimensions via .mean(dim=('chain','draw'))
# TODO: Compute posterior SD over the same dimensions via .std(dim=('chain','draw'))
# TODO: Build and return BayesianPosterior dataclass

### 8.7 Execute and report

Run all Bayesian model steps in sequence and print a self-contained pass/fail summary. The notebook should not proceed to scoring if `ConvergenceReport.passed` is `False` without an explicit human acknowledgement.

In [ ]:
# TODO: Prepare arrays from TRAIN_WEIGHTED_DF: home_idx, away_idx, home_goals, away_goals, is_neutral, weights
# TODO: BAYES_MODEL = build_bayesian_model(home_idx, away_idx, home_goals, away_goals, is_neutral, weights, len(CORE_TEAMS), CFG.bayesian)
# TODO: BAYES_IDATA_BACKTEST: az.InferenceData = fit_bayesian_model(BAYES_MODEL, CFG.bayesian)
# TODO: CONVERGENCE_REPORT: ConvergenceReport = check_convergence(BAYES_IDATA_BACKTEST, CFG.bayesian)
# TODO: Print ConvergenceReport; raise a RuntimeWarning if passed is False
# TODO: PPC_SUMMARY = posterior_predictive_check(BAYES_IDATA_BACKTEST, home_goals, away_goals)
# TODO: BAYES_POSTERIOR_BACKTEST: BayesianPosterior = extract_posterior_means(BAYES_IDATA_BACKTEST, sorted(CORE_TEAMS))
# TODO: Print attack/defense posterior mean and SD for FIXTURE_HOME_TEAM and FIXTURE_AWAY_TEAM

---

## Section 9 — Optuna Hyperparameter Search

Tunes the XGBoost hyperparameters using nested expanding-window cross-validation **entirely within `TRAIN_FEATURES`**. The outer `TEST_FEATURES` frame is never passed to Optuna or seen during the search — its use for final evaluation in Section 12 is not compromised.

**Objective:** minimize mean Poisson deviance (log loss of the Poisson distribution) averaged over CV folds, weighting each match by `match_weight`. The Poisson deviance is appropriate here because XGBoost uses `count:poisson` objective, so the CV metric should be consistent with the training objective.

### 9.1 `expanding_window_splits`

Generates `n_splits` chronological folds from the training frame. Each fold's training window expands from the global start date to a fold-specific cutoff; the validation window covers the next contiguous block. No fold's validation window is allowed to overlap `TEST_FEATURES`.

In [ ]:
# TODO: Implement expanding_window_splits(df: pl.DataFrame, n_splits: int, min_train_fraction: float) -> Iterator[tuple[pl.DataFrame, pl.DataFrame]]
# TODO: Divide the date range of df into (n_splits + 1) equal-size blocks
# TODO: Fold i: train = all rows before block i+1, validation = block i+1
# TODO: Skip folds where train fraction < min_train_fraction
# TODO: Yield (train_fold, val_fold) tuples

### 9.2 `optuna_objective`

Defines the Optuna trial objective. For each trial, it samples hyperparameters from `XGBoostConfig.search_space`, trains home and away goal models on each CV fold, predicts on the validation fold, and returns the mean weighted Poisson deviance.

In [ ]:
# TODO: Implement optuna_objective(trial: optuna.Trial, train_features: pl.DataFrame, feature_cols: list[str], weight_col: str, config: OptunaConfig, xgb_config: XGBoostConfig) -> float
# TODO: Sample: n_estimators, max_depth, learning_rate, subsample, colsample_bytree, reg_alpha, reg_lambda from search_space
# TODO: For each fold from expanding_window_splits: fit home/away models, predict, compute mean Poisson deviance on val
# TODO: Return mean deviance across folds (lower is better)

### 9.3 Run Optuna study

Creates a study with a reproducible seed and runs for `OptunaConfig.n_trials` trials or until `timeout_seconds` is reached, whichever comes first.

In [ ]:
# TODO: Implement run_optuna_search(train_features: pl.DataFrame, feature_cols: list[str], weight_col: str, config: OptunaConfig, xgb_config: XGBoostConfig) -> dict[str, float | int | str]
# TODO: Create optuna.create_study(direction='minimize', sampler=TPESampler(seed=config.sampler_seed))
# TODO: study.optimize with n_trials and timeout
# TODO: Return study.best_params merged with XGBoostConfig.fixed_params
# TODO: BEST_HYPERPARAMS: dict[str, float | int | str] = run_optuna_search(TRAIN_FEATURES, CFG.features.feature_columns, 'match_weight', CFG.optuna, CFG.xgboost)

### 9.4 Inspect best parameters

Print best trial value, best hyperparameters, and an importance plot for the Optuna hyperparameter importances (if more than 20 trials were completed).

In [ ]:
# TODO: Print best trial Poisson deviance and BEST_HYPERPARAMS
# TODO: If study has >= 20 trials: plot optuna.visualization.plot_param_importances(study) using matplotlib backend

---

## Section 10 — XGBoost Fit (Backtest)

Trains two independent XGBoost Poisson regressors on `TRAIN_FEATURES` using `BEST_HYPERPARAMS` from Section 9. One model predicts home goals; the other predicts away goals. Sample weights (`match_weight`) are passed to `fit()` so that recent, high-importance matches receive more influence over the final parameters.

**Note:** XGBoost's `count:poisson` objective maps naturally to goal counts — it constrains predictions to be non-negative and uses a log link, consistent with the Poisson assumption used by both other models.

### 10.1 `train_xgboost_goal_models`

In [ ]:
# TODO: Implement train_xgboost_goal_models(train_df: pl.DataFrame, feature_cols: list[str], weight_col: str, hyperparameters: dict[str, float | int | str]) -> tuple[xgb.XGBRegressor, xgb.XGBRegressor]
# TODO: Extract X = train_df.select(feature_cols).to_numpy(), w = train_df[weight_col].to_numpy()
# TODO: y_home = train_df['home_score'].to_numpy(), y_away = train_df['away_score'].to_numpy()
# TODO: Fit home_model.fit(X, y_home, sample_weight=w); fit away_model.fit(X, y_away, sample_weight=w)
# TODO: Return (home_model, away_model)
# TODO: XGB_HOME_MODEL, XGB_AWAY_MODEL = train_xgboost_goal_models(TRAIN_FEATURES, CFG.features.feature_columns, 'match_weight', BEST_HYPERPARAMS)

### 10.2 `predict_goal_rates`

In [ ]:
# TODO: Implement predict_goal_rates(home_model: xgb.XGBRegressor, away_model: xgb.XGBRegressor, features: pl.DataFrame, feature_cols: list[str]) -> tuple[np.ndarray, np.ndarray]
# TODO: Return (home_model.predict(X), away_model.predict(X)) where X = features.select(feature_cols).to_numpy()

### 10.3 Feature importance

Print and plot feature importances for both models. Attack/defense DC features and recent form should dominate; if neutral-venue flag unexpectedly dominates, this warrants investigation.

In [ ]:
# TODO: Print sorted feature importances for XGB_HOME_MODEL and XGB_AWAY_MODEL
# TODO: Plot side-by-side horizontal bar charts of importances for both models

---

## Section 11 — Backtest Scoring

Generates a `(K+1) × (K+1)` scoreline probability matrix for every match in `TEST_DF` using each of the three models. The output is three matrix stacks of shape `(n_test, K+1, K+1)` — one per model — used for metric computation in Section 12.

**Bayesian backtest approximation:** for the test set (many rows), we use the posterior-mean attack/defense parameters rather than integrating over the full posterior. This is an accepted approximation for backtest scoring; the full posterior-predictive average is reserved for the headline fixture in Section 15, where it matters most.

### 11.1 Scoring utility functions

In [ ]:
# TODO: Implement poisson_score_matrix(lam: float, mu: float, max_goals: int, rho: float) -> np.ndarray  — single-fixture matrix, shape (max_goals+1, max_goals+1)
# TODO: Implement batch_poisson_score_matrices(lam: np.ndarray, mu: np.ndarray, max_goals: int, rho: float) -> np.ndarray  — vectorized, shape (n, max_goals+1, max_goals+1)
# TODO: Implement score_matrix_to_outcome_probs(matrix: np.ndarray) -> OutcomeProbabilities
# TODO: Implement top_n_scorelines(matrix: np.ndarray, n: int) -> list[tuple[tuple[int, int], float]]

### 11.2 Dataclass: `OutcomeProbabilities`

In [ ]:
# TODO: Define OutcomeProbabilities dataclass: p_home_win: float, p_draw: float, p_away_win: float
# TODO: Add a __str__ method that formats the three probabilities as percentages

### 11.3 Dixon-Coles test matrices

Uses `DC_RATINGS_BACKTEST` to compute expected goal rates for each test fixture, then applies `batch_poisson_score_matrices` with the fitted `rho` correction.

In [ ]:
# TODO: Compute lambda_dc and mu_dc for each TEST_DF row using DC_RATINGS_BACKTEST attack/defense/home_advantage/intercept
# TODO: Account for is_neutral flag: zero out home_advantage for neutral-venue matches
# TODO: DC_TEST_MATRICES: np.ndarray = batch_poisson_score_matrices(lambda_dc, mu_dc, CFG.evaluation.max_goals, DC_RATINGS_BACKTEST.rho)
# TODO: Print shape and a sanity check: first matrix should sum to approximately 1.0

### 11.4 Bayesian test matrices

Uses posterior-mean attack/defense from `BAYES_POSTERIOR_BACKTEST` to compute expected goal rates for each test fixture.

In [ ]:
# TODO: Implement bayesian_rate_means_batch(posterior: BayesianPosterior, home_idx: np.ndarray, away_idx: np.ndarray, is_neutral: np.ndarray, home_advantage_mean: float, intercept_mean: float) -> tuple[np.ndarray, np.ndarray]
# TODO: Compute lambda_bayes and mu_bayes for each TEST_DF row using posterior means
# TODO: BAYES_TEST_MATRICES: np.ndarray = batch_poisson_score_matrices(lambda_bayes, mu_bayes, CFG.evaluation.max_goals, rho=0.0)
# TODO: Note: rho=0.0 for Bayesian matrices — the model does not include the tau correction

### 11.5 XGBoost test matrices

In [ ]:
# TODO: lambda_xgb, mu_xgb = predict_goal_rates(XGB_HOME_MODEL, XGB_AWAY_MODEL, TEST_FEATURES, CFG.features.feature_columns)
# TODO: XGB_TEST_MATRICES: np.ndarray = batch_poisson_score_matrices(lambda_xgb, mu_xgb, CFG.evaluation.max_goals, rho=0.0)

---

## Section 12 — Backtest Evaluation

Computes metrics for each model and for two trivial baselines. Every model must beat both baselines to justify its complexity.

**Metrics computed:**

- `exact_score_accuracy` — proportion of test matches where the modal predicted scoreline matches the actual scoreline
- `outcome_accuracy` — 1X2 classification accuracy
- `log_loss` — multiclass negative log-likelihood over 1X2 probabilities
- `brier_score` — mean squared probability error over 1X2 probabilities
- `home_goals_mae`, `away_goals_mae` — mean absolute error on expected-goal predictions
- `home_goals_rmse`, `away_goals_rmse` — RMSE on expected-goal predictions

**Baselines:**

1. **Home-bias baseline** — constant 1X2 probabilities computed from the training-set home-win / draw / away-win proportions, ignoring all team identities.
2. **Mean-goals baseline** — predict each team's overall training-set average goals, independent of opponent.

### 12.1 `ModelMetrics` dataclass

In [ ]:
# TODO: Define ModelMetrics dataclass: model_name: str, exact_score_accuracy: float, outcome_accuracy: float, log_loss: float, brier_score: float, home_goals_mae: float, away_goals_mae: float, home_goals_rmse: float, away_goals_rmse: float, n_evaluated: int

### 12.2 `evaluate_predictions`

Takes a matrix stack and actual goal arrays and returns a `ModelMetrics` object.

In [ ]:
# TODO: Implement evaluate_predictions(matrices: np.ndarray, actual_home: np.ndarray, actual_away: np.ndarray, lam: np.ndarray, mu: np.ndarray, model_name: str) -> ModelMetrics
# TODO: Compute each metric defined in EvaluationConfig.metrics
# TODO: For log_loss and brier_score: collapse matrices to 1X2 probabilities first via score_matrix_to_outcome_probs (vectorized)

### 12.3 `evaluate_baselines`

In [ ]:
# TODO: Implement evaluate_baselines(train_df: pl.DataFrame, test_df: pl.DataFrame) -> dict[str, ModelMetrics]
# TODO: Home-bias baseline: 1X2 proportions from train_df applied uniformly to every test match
# TODO: Mean-goals baseline: mean home and away goals from train_df as lambda/mu for every test match
# TODO: Return dict keyed by baseline name

### 12.4 Execute evaluation

In [ ]:
# TODO: actual_home = TEST_DF['home_score'].to_numpy(); actual_away = TEST_DF['away_score'].to_numpy()
# TODO: DC_METRICS = evaluate_predictions(DC_TEST_MATRICES, actual_home, actual_away, lambda_dc, mu_dc, 'Dixon-Coles')
# TODO: BAYES_METRICS = evaluate_predictions(BAYES_TEST_MATRICES, actual_home, actual_away, lambda_bayes, mu_bayes, 'Bayesian')
# TODO: XGB_METRICS = evaluate_predictions(XGB_TEST_MATRICES, actual_home, actual_away, lambda_xgb, mu_xgb, 'XGBoost')
# TODO: BASELINE_METRICS = evaluate_baselines(TRAIN_DF, TEST_DF)
# TODO: ALL_METRICS: dict[str, ModelMetrics] = {'Dixon-Coles': DC_METRICS, 'Bayesian': BAYES_METRICS, 'XGBoost': XGB_METRICS, **BASELINE_METRICS}

### 12.5 Metrics table and calibration

Display all metrics in a formatted table. A model that does not beat both baselines on both `log_loss` and `outcome_accuracy` is flagged with a visible warning.

In [ ]:
# TODO: Print ALL_METRICS as a formatted Polars DataFrame sorted by log_loss ascending
# TODO: Assert each real model beats both baselines on log_loss; print a visible WARNING if any do not
# TODO: Implement calibration_curve_1x2(predicted_probs: np.ndarray, actual_outcomes: np.ndarray, n_bins: int) -> pl.DataFrame
# TODO: Compute calibration curves for each model and store as CALIBRATION_CURVES dict

---

## Section 13 — Ensemble Weight Derivation

Derives the ensemble combination weights from the three models' backtest log-loss scores using a softmax over negative log loss. `EnsembleConfig.temperature` controls concentration: lower temperature gives more weight to the best-performing model; higher temperature gives a more uniform blend.

These weights are then used in Section 15 to combine the three production-model matrices for the headline fixture.

### 13.1 `compute_ensemble_weights`

In [ ]:
# TODO: Implement compute_ensemble_weights(backtest_metrics: dict[str, ModelMetrics], temperature: float) -> dict[str, float]
# TODO: Extract log_loss from each real model (exclude baseline entries)
# TODO: weight_m = exp(-log_loss_m / temperature) / sum_m' exp(-log_loss_m' / temperature)
# TODO: Return dict[model_name -> weight], values summing to 1.0

### 13.2 Compute and store

In [ ]:
# TODO: ENSEMBLE_WEIGHTS: dict[str, float] = compute_ensemble_weights(ALL_METRICS, CFG.ensemble.temperature)
# TODO: Print weights and the log-loss values they were derived from
# TODO: Store into CFG by constructing updated EnsembleConfig; or store separately as FINAL_ENSEMBLE_CONFIG

---

## Section 14 — Production Refit

All three models are refit on the **full played history** (`CORE_MATCH_DF`, which includes both the backtest train and test windows) to produce the best possible current ratings for predicting the headline fixture.

The same functions from Sections 5-10 are called again — nothing is reimplemented. The only change is the input frame and the reference date for time-decay weights. This strict reuse is intentional: it confirms that the backtest functions generalize and prevents silent divergence between the evaluated and deployed models.

### 14.1 Full-history weights

In [ ]:
# TODO: FULL_WEIGHTED_DF = compute_match_weights(CORE_MATCH_DF, PRODUCTION_REFERENCE_DATE, CFG.weighting)
# TODO: Print match count and weight summary for FULL_WEIGHTED_DF

### 14.2 Production Dixon-Coles fit

In [ ]:
# TODO: DC_RATINGS_PROD: DixonColesRatings = fit_dixon_coles(FULL_WEIGHTED_DF, sorted(CORE_TEAMS), CFG.dixon_coles)
# TODO: Print production home_advantage, intercept, rho
# TODO: Print attack/defense for FIXTURE_HOME_TEAM and FIXTURE_AWAY_TEAM
# TODO: Compare production vs backtest attack values to sense-check consistency (no assertion — just informational)

### 14.3 Production Bayesian fit

Full NUTS refit on all available match data. Convergence is checked again — there is no assumption that the production model converges just because the backtest model did.

In [ ]:
# TODO: Prepare arrays from FULL_WEIGHTED_DF: home_idx_prod, away_idx_prod, home_goals_prod, away_goals_prod, is_neutral_prod, weights_prod
# TODO: BAYES_MODEL_PROD = build_bayesian_model(home_idx_prod, away_idx_prod, home_goals_prod, away_goals_prod, is_neutral_prod, weights_prod, len(CORE_TEAMS), CFG.bayesian)
# TODO: BAYES_IDATA_PROD: az.InferenceData = fit_bayesian_model(BAYES_MODEL_PROD, CFG.bayesian)
# TODO: CONVERGENCE_REPORT_PROD: ConvergenceReport = check_convergence(BAYES_IDATA_PROD, CFG.bayesian)
# TODO: Print production convergence report; warn if not passed
# TODO: BAYES_POSTERIOR_PROD: BayesianPosterior = extract_posterior_means(BAYES_IDATA_PROD, sorted(CORE_TEAMS))

### 14.4 Production XGBoost fit

Feature engineering is re-run on `CORE_MATCH_DF` for current-form feature values, but `compute_current_form` (unshifted) is used only for predicting the fixture in Section 15 — the production XGBoost models are still trained on the shifted rolling features to remain consistent with the backtest.

In [ ]:
# TODO: FEATURE_DF_PROD = add_dixon_coles_features(add_rolling_form_features(CORE_MATCH_DF, CFG.features.rolling_window), DC_RATINGS_PROD)
# TODO: FULL_FEATURES = FEATURE_DF_PROD.join(FULL_WEIGHTED_DF.select(['match_id','match_weight']), on='match_id', how='left')
# TODO: XGB_HOME_MODEL_PROD, XGB_AWAY_MODEL_PROD = train_xgboost_goal_models(FULL_FEATURES, CFG.features.feature_columns, 'match_weight', BEST_HYPERPARAMS)

---

## Section 15 — Headline Fixture Prediction

Generates one scoreline matrix per production model for the headline fixture (Argentina vs Austria, 2026-06-22, neutral venue), then combines them into the ensemble matrix using `ENSEMBLE_WEIGHTS`.

**Bayesian treatment here differs from the backtest:** rather than using posterior-mean parameters, we draw the full posterior-predictive distribution by averaging the Poisson matrix over all NUTS posterior samples. This gives a genuine uncertainty-aware prediction rather than a plug-in approximation.

### 15.1 Current form snapshot

Compute each team's form over their `rolling_window` most recent matches using `compute_current_form` (unshifted — this is the production-time form snapshot, not a training feature row).

In [ ]:
# TODO: CURRENT_FORM_DF = compute_current_form(CORE_MATCH_DF, CFG.features.rolling_window)
# TODO: Print form row for FIXTURE_HOME_TEAM and FIXTURE_AWAY_TEAM

### 15.2 `build_fixture_feature_row`

Assembles the single-row Polars DataFrame used as XGBoost input for the fixture.

In [ ]:
# TODO: Implement build_fixture_feature_row(home_team: str, away_team: str, is_neutral: bool, ratings: DixonColesRatings, current_form: pl.DataFrame, feature_cols: list[str]) -> pl.DataFrame
# TODO: Pull DC attack/defense from ratings for home and away team
# TODO: Pull form_goals_for, form_goals_against, form_points for both teams from current_form
# TODO: Set is_neutral to int(is_neutral)
# TODO: Return a single-row DataFrame with exactly CFG.features.feature_columns as column names
# TODO: FIXTURE_FEATURE_ROW = build_fixture_feature_row(FIXTURE_HOME_TEAM, FIXTURE_AWAY_TEAM, CFG.fixture.is_neutral_venue, DC_RATINGS_PROD, CURRENT_FORM_DF, CFG.features.feature_columns)
# TODO: Print FIXTURE_FEATURE_ROW for manual inspection

### 15.3 Per-model scoreline matrices

In [ ]:
# TODO: # Dixon-Coles production matrix
# TODO: Compute lambda_dc_prod and mu_dc_prod for the fixture from DC_RATINGS_PROD (zero home_advantage because is_neutral=True)
# TODO: DC_FIXTURE_MATRIX: np.ndarray = poisson_score_matrix(lambda_dc_prod, mu_dc_prod, CFG.evaluation.max_goals, DC_RATINGS_PROD.rho)
# TODO: # Bayesian posterior-predictive matrix
# TODO: Implement bayesian_rate_samples(idata: az.InferenceData, team_index: dict[str, int], home_team: str, away_team: str, is_neutral: bool, home_advantage_samples: np.ndarray, intercept_samples: np.ndarray) -> tuple[np.ndarray, np.ndarray]
# TODO: Implement posterior_predictive_score_matrix(lam_samples: np.ndarray, mu_samples: np.ndarray, max_goals: int) -> np.ndarray  — average of per-draw Poisson matrices via vectorized outer product
# TODO: lambda_bayes_samples, mu_bayes_samples = bayesian_rate_samples(BAYES_IDATA_PROD, TEAM_INDEX, FIXTURE_HOME_TEAM, FIXTURE_AWAY_TEAM, CFG.fixture.is_neutral_venue, ...)
# TODO: BAYES_FIXTURE_MATRIX: np.ndarray = posterior_predictive_score_matrix(lambda_bayes_samples, mu_bayes_samples, CFG.evaluation.max_goals)
# TODO: # XGBoost production matrix
# TODO: lambda_xgb_prod, mu_xgb_prod = predict_goal_rates(XGB_HOME_MODEL_PROD, XGB_AWAY_MODEL_PROD, FIXTURE_FEATURE_ROW, CFG.features.feature_columns)
# TODO: XGB_FIXTURE_MATRIX: np.ndarray = poisson_score_matrix(float(lambda_xgb_prod[0]), float(mu_xgb_prod[0]), CFG.evaluation.max_goals, rho=0.0)

### 15.4 Ensemble matrix and prediction summary

In [ ]:
# TODO: Implement combine_score_matrices(matrices: dict[str, np.ndarray], weights: dict[str, float]) -> np.ndarray
# TODO: Implement prediction_entropy(matrix: np.ndarray) -> float  — -sum(p * log(p + eps)) over all cells
# TODO: FIXTURE_MATRICES: dict[str, np.ndarray] = {'Dixon-Coles': DC_FIXTURE_MATRIX, 'Bayesian': BAYES_FIXTURE_MATRIX, 'XGBoost': XGB_FIXTURE_MATRIX}
# TODO: ENSEMBLE_MATRIX: np.ndarray = combine_score_matrices(FIXTURE_MATRICES, ENSEMBLE_WEIGHTS)
# TODO: Define FixturePrediction dataclass: home_team, away_team, match_date, per_model_matrices, ensemble_matrix, ensemble_weights, outcome, top_scorelines, entropy
# TODO: PREDICTION = FixturePrediction(home_team=FIXTURE_HOME_TEAM, away_team=FIXTURE_AWAY_TEAM, match_date=CFG.fixture.match_date, per_model_matrices=FIXTURE_MATRICES, ensemble_matrix=ENSEMBLE_MATRIX, ensemble_weights=ENSEMBLE_WEIGHTS, outcome=score_matrix_to_outcome_probs(ENSEMBLE_MATRIX), top_scorelines=top_n_scorelines(ENSEMBLE_MATRIX, n=10), entropy=prediction_entropy(ENSEMBLE_MATRIX))
# TODO: Print prediction summary: outcome probs, top-3 scorelines, entropy, per-model ensemble weights

---

## Section 16 — Visualization: Scoreline Heatmap

Renders the ensemble scoreline probability matrix as an annotated heatmap. Each cell `[i, j]` shows the probability of home team scoring `i` goals and away team scoring `j` goals. Diagonal cells (draws) are highlighted with a distinct border. Probabilities below 0.5% are suppressed to reduce visual noise.

The three per-model matrices are optionally shown in a smaller side-by-side panel below to allow direct comparison of each model's predictive distribution.

### 16.1 `plot_score_heatmap`

In [ ]:
# TODO: Implement plot_score_heatmap(matrix: np.ndarray, home_team: str, away_team: str, max_goals_display: int, title_suffix: str) -> matplotlib.figure.Figure
# TODO: Use seaborn.heatmap with annotate=True (formatted as percentages), cmap='YlOrRd'
# TODO: x-axis: away goals 0..max_goals_display; y-axis: home goals 0..max_goals_display
# TODO: Highlight diagonal cells (draws) with a thin border using ax.patches
# TODO: Suppress cells below 0.5% probability in the annotation text
# TODO: Label axes with team names and add a colorbar labelled 'Probability'

### 16.2 Render ensemble heatmap

In [ ]:
# TODO: FIG_HEATMAP = plot_score_heatmap(ENSEMBLE_MATRIX, FIXTURE_HOME_TEAM, FIXTURE_AWAY_TEAM, max_goals_display=6, title_suffix='(Ensemble)')
# TODO: plt.tight_layout(); plt.show()

### 16.3 Per-model comparison panel (optional)

In [ ]:
# TODO: fig, axes = plt.subplots(1, 3, figsize=(18, 5))
# TODO: For each (model_name, matrix) in PREDICTION.per_model_matrices.items(): plot heatmap on corresponding axis
# TODO: Add a shared colorbar and a figure-level title
# TODO: plt.tight_layout(); plt.show()

---

## Section 17 — Visualization: 1X2 Outcome Bar Chart

Collapses the ensemble scoreline matrix into three outcome probabilities (home win / draw / away win) and renders them as a horizontal bar chart. Each model's 1X2 probabilities are shown as lighter bars behind the ensemble values to make the contribution of each model visible.

### 17.1 `plot_outcome_probabilities`

In [ ]:
# TODO: Implement plot_outcome_probabilities(outcome: OutcomeProbabilities, home_team: str, away_team: str, per_model_outcomes: dict[str, OutcomeProbabilities] | None) -> matplotlib.figure.Figure
# TODO: Three bars: home win (blue), draw (grey), away win (red) — widths proportional to probability
# TODO: Annotate each bar with the probability as a percentage
# TODO: If per_model_outcomes is provided: overlay lighter translucent bars for each model behind the ensemble bars
# TODO: x-axis: 0 to 1.0 (probability); y-axis: outcome labels

### 17.2 Render

In [ ]:
# TODO: per_model_outcomes = {name: score_matrix_to_outcome_probs(m) for name, m in PREDICTION.per_model_matrices.items()}
# TODO: FIG_OUTCOMES = plot_outcome_probabilities(PREDICTION.outcome, FIXTURE_HOME_TEAM, FIXTURE_AWAY_TEAM, per_model_outcomes)
# TODO: plt.tight_layout(); plt.show()

---

## Section 18 — Visualization: Top-10 Scorelines Bar Chart

Ranks the ten most probable exact scorelines from the ensemble matrix and renders them as a vertical bar chart. Bars are colour-coded by outcome type: blue for home-team wins, grey for draws, red for away-team wins.

### 18.1 `plot_top_n_scorelines`

In [ ]:
# TODO: Implement plot_top_n_scorelines(matrix: np.ndarray, home_team: str, away_team: str, n: int) -> matplotlib.figure.Figure
# TODO: Extract top-n scorelines via top_n_scorelines(matrix, n)
# TODO: Format x-axis labels as 'H-A' strings (e.g. '2-1', '0-0')
# TODO: Colour each bar: blue if home_goals > away_goals, grey if equal, red if away_goals > home_goals
# TODO: Annotate each bar with its probability as a percentage
# TODO: y-axis: probability from 0 to max_prob * 1.15 to leave room for annotations

### 18.2 Render

In [ ]:
# TODO: FIG_TOP10 = plot_top_n_scorelines(ENSEMBLE_MATRIX, FIXTURE_HOME_TEAM, FIXTURE_AWAY_TEAM, n=10)
# TODO: plt.tight_layout(); plt.show()

---

## Section 19 — Diagnostic Plots

Supporting plots that allow the reader to audit model quality independently of the headline fixture prediction.

### 19.1 MCMC trace and posterior plots

ArviZ trace plots for `home_advantage`, `intercept`, and the attack/defense parameters for the two headline fixture teams. Confirms chain mixing and stationarity visually.

In [ ]:
# TODO: az.plot_trace(BAYES_IDATA_PROD, var_names=['home_advantage', 'intercept'], compact=True)
# TODO: plt.tight_layout(); plt.show()
# TODO: Plot posterior distributions for attack[FIXTURE_HOME_TEAM], defense[FIXTURE_HOME_TEAM], attack[FIXTURE_AWAY_TEAM], defense[FIXTURE_AWAY_TEAM] using az.plot_posterior

### 19.2 Calibration curves

For each model, plot predicted 1X2 probability buckets against observed outcome frequency. A well-calibrated model should produce points close to the diagonal.

In [ ]:
# TODO: Implement plot_calibration_curve(curves: dict[str, pl.DataFrame]) -> matplotlib.figure.Figure
# TODO: Plot calibration_curve_1x2 output for each model on the same axes
# TODO: Add the diagonal reference line; label each model's curve
# TODO: plt.tight_layout(); plt.show()

### 19.3 Backtest metrics comparison table

In [ ]:
# TODO: Implement plot_backtest_metrics_table(metrics: dict[str, ModelMetrics]) -> matplotlib.figure.Figure
# TODO: Render as a Matplotlib table with cells colour-coded by relative performance (green = best, red = worst) per metric
# TODO: Baseline rows are visually distinguished (e.g. italic or lighter background)
# TODO: plt.tight_layout(); plt.show()

---

## Section 20 — Summary & Limitations

### Prediction Summary

> This section is populated at runtime by a code cell that formats `PREDICTION` as prose.

### Known Simplifications

1. **Dixon-Coles features are train-window-wide, not per-row rolling.** The DC ratings used as XGBoost features are fit once on the entire training window, then used as a static feature for every training row including early ones. A fully leak-free alternative would refit DC on a strictly trailing window per match, which is computationally expensive and out of scope for this implementation.

2. **Bayesian backtest uses posterior-mean rates, not full posterior predictive.** The backtest metrics for the Bayesian model are computed from posterior-mean attack/defense parameters rather than the average of per-posterior-draw matrices. This is a tractability approximation; the full posterior-predictive treatment is applied only to the headline fixture.

3. **XGBoost matrices have no `tau` correction.** The Dixon-Coles low-score correlation correction is applied only to the DC model's matrices because XGBoost's predictions are independent Poisson means with no joint low-score structure estimated. A future extension could post-multiply XGBoost matrices by the DC fitted `rho`.

4. **Team ratings assume a stationary 2018-present process.** The time-decay weighting partially addresses non-stationarity, but regime changes (coaching, formation, player turnover) are not explicitly modelled. The walk-forward validation is the primary tool for assessing whether this matters for a given fixture.

5. **Injury and squad availability are not modelled.** The system operates entirely on historical match-level results. No player-level data is used, and no adjustment is made for missing key players in the headline fixture.

6. **Core-team minimum-match threshold excludes some nations.** Teams below `TeamFilterConfig.min_matches` receive no rating and cannot be predicted for. This affects primarily Pacific and Caribbean nations with sparse international schedules.

### Risk Register (Summary)

| Risk | Mitigation |
|---|---|
| Random train/test split leakage | Chronological split enforced by a single function |
| Rolling-form feature leakage | Shift-before-rolling, spot-check test in Section 7 |
| Tournament-weight mismatching | Exact-string lookup table; accent-aware keys; warning on unknown names |
| Bayesian non-convergence | Automatic `ConvergenceReport` with pass/fail; notebook warns on failure |
| Optuna peek at test set | Search operates only within `TRAIN_FEATURES` |
| Recency-weight reference date | Explicit `reference_date` parameter; never hardcoded |
| Upstream schema change | `validate_schema` fails immediately with a descriptive diff |
| Stochastic irreproducibility | Every seed is a named config field; no bare `np.random` calls |

In [ ]:
# TODO: Format PREDICTION as a prose summary: fixture name, date, outcome probabilities (%), modal scoreline, entropy
# TODO: Print the formatted summary